# Debugging the N-body integrator for Experiment 1

The orbital demo in `stress_tests.ipynb` shows inner planets (small $r$) are
**unstable**, while outer planets (large $r$) don't drift even though they should
barely be bound.

**Root causes**:
1. The integrator uses first-order symplectic Euler (kick-drift), which is only $O(\Delta t)$
   accurate.  This causes systematic energy drift, especially for fast inner orbits.
2. The timestep is fixed.  Inner planets with orbital periods $\sim 1$ need $\Delta t \ll 0.01$;
   outer planets with periods $\sim 200$ need long simulation times to see any effect.
3. The FieldSampler uses an adaptive grid (`recommend_grid_params`), but the direct
   N-body integrator has no equivalent — it uses a single global $\Delta t$.

**Fix**: switch to velocity Verlet (second-order leapfrog), which conserves energy
to $O(\Delta t^2)$ and is symplectic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'font.size': 13})

G_scr = 1.0
eps_soft = 0.1

def screened_inv_sq_force(r_vec, r_mag, m_i, m_j, lam=5.0):
    """F = G * m_i * m_j / r^2 * exp(-r/lambda), attractive, with softening."""
    F_mag = G_scr * m_i * m_j / (r_mag**2 + eps_soft**2) * np.exp(-r_mag / lam)
    return -F_mag * r_vec / r_mag

## Integrator comparison: symplectic Euler vs velocity Verlet

For a single planet orbiting a sun, compare the two schemes at the same $\Delta t$.
Symplectic Euler (first-order) drifts in energy; velocity Verlet (second-order) conserves it.

In [ ]:
def compute_accel_pair(pos_sun, pos_planet, M_sun, m_planet, lam):
    """Acceleration on the planet from the sun (ignores back-reaction for clarity)."""
    r_vec = pos_planet - pos_sun
    r_mag = np.linalg.norm(r_vec)
    F_mag = G_scr * M_sun * m_planet / (r_mag**2 + eps_soft**2) * np.exp(-r_mag / lam)
    force = -F_mag * r_vec / r_mag
    return force / m_planet

def run_euler(r0, v_circ, M_sun, m_planet, lam, dt, n_steps):
    """Symplectic Euler: kick then drift. First order."""
    pos = np.array([r0, 0.0])
    vel = np.array([0.0, v_circ])
    radii = np.zeros(n_steps)
    energies = np.zeros(n_steps)
    for i in range(n_steps):
        acc = compute_accel_pair(np.zeros(2), pos, M_sun, m_planet, lam)
        vel += dt * acc        # kick
        pos += dt * vel        # drift
        radii[i] = np.linalg.norm(pos)
        r = radii[i]
        # KE + PE (PE from integrating F dr)
        KE = 0.5 * m_planet * np.dot(vel, vel)
        energies[i] = KE  # PE is complicated for this force; just track KE as proxy
    return radii, energies

def run_verlet(r0, v_circ, M_sun, m_planet, lam, dt, n_steps):
    """Velocity Verlet (leapfrog): half-kick, drift, half-kick. Second order."""
    pos = np.array([r0, 0.0])
    vel = np.array([0.0, v_circ])
    radii = np.zeros(n_steps)
    energies = np.zeros(n_steps)
    acc = compute_accel_pair(np.zeros(2), pos, M_sun, m_planet, lam)
    for i in range(n_steps):
        vel += 0.5 * dt * acc       # half kick
        pos += dt * vel             # full drift
        acc_new = compute_accel_pair(np.zeros(2), pos, M_sun, m_planet, lam)
        vel += 0.5 * dt * acc_new   # half kick
        acc = acc_new
        radii[i] = np.linalg.norm(pos)
        KE = 0.5 * m_planet * np.dot(vel, vel)
        energies[i] = KE
    return radii, energies

# Test at several radii
M_sun = 50.0
m_planet = 0.1
lam = 5.0
dt = 0.002
n_steps = 10000

test_radii = [1.0, 2.0, 5.0, 10.0, 15.0]

fig, axes = plt.subplots(2, len(test_radii), figsize=(4*len(test_radii), 8), sharex='col')

for col, r0 in enumerate(test_radii):
    F_r = G_scr * M_sun * m_planet / r0**2 * np.exp(-r0 / lam)
    v_circ = np.sqrt(F_r * r0 / m_planet)
    T_orb = 2 * np.pi * r0 / v_circ

    r_euler, e_euler = run_euler(r0, v_circ, M_sun, m_planet, lam, dt, n_steps)
    r_verlet, e_verlet = run_verlet(r0, v_circ, M_sun, m_planet, lam, dt, n_steps)

    times = np.arange(n_steps) * dt
    n_orbits = times / T_orb

    ax = axes[0, col]
    ax.plot(n_orbits, r_euler, 'r-', lw=0.5, alpha=0.7, label='Euler')
    ax.plot(n_orbits, r_verlet, 'b-', lw=0.5, alpha=0.7, label='Verlet')
    ax.axhline(r0, color='gray', ls=':', lw=0.5)
    ax.set_title(f'$r_0 = {r0}$\n$T = {T_orb:.1f}$, $\\Delta t/T = {dt/T_orb:.4f}$')
    ax.set_ylabel('$r(t)$')
    if col == 0:
        ax.legend(fontsize=7)

    ax = axes[1, col]
    ax.plot(n_orbits, (r_euler - r0) / r0 * 100, 'r-', lw=0.5, alpha=0.7)
    ax.plot(n_orbits, (r_verlet - r0) / r0 * 100, 'b-', lw=0.5, alpha=0.7)
    ax.axhline(0, color='gray', ls=':', lw=0.5)
    ax.set_xlabel('Orbits')
    ax.set_ylabel('$\\Delta r / r_0$ (%)')

plt.suptitle(f'Euler vs Verlet at dt={dt} (lam={lam})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
for r0 in test_radii:
    F_r = G_scr * M_sun * m_planet / r0**2 * np.exp(-r0 / lam)
    v_circ = np.sqrt(F_r * r0 / m_planet)
    T_orb = 2 * np.pi * r0 / v_circ
    steps_per_orbit = T_orb / dt
    print(f"  r={r0:5.1f}: T_orb={T_orb:8.2f}, steps/orbit={steps_per_orbit:8.0f}, "
          f"  v_circ={v_circ:.3f}, F={F_r:.4f}")

## Timestep convergence

For the problematic $r=1$ orbit, reduce $\Delta t$ until both integrators converge.
Verlet should converge much faster (second order vs first order).

In [ ]:
r0_test = 1.0
F_r = G_scr * M_sun * m_planet / r0_test**2 * np.exp(-r0_test / lam)
v_circ_test = np.sqrt(F_r * r0_test / m_planet)
T_orb_test = 2 * np.pi * r0_test / v_circ_test

dts = [0.005, 0.002, 0.001, 0.0005, 0.0002]
n_orbits_target = 20
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for dt_test in dts:
    n_steps_test = int(n_orbits_target * T_orb_test / dt_test)
    r_e, _ = run_euler(r0_test, v_circ_test, M_sun, m_planet, lam, dt_test, n_steps_test)
    r_v, _ = run_verlet(r0_test, v_circ_test, M_sun, m_planet, lam, dt_test, n_steps_test)
    times_test = np.arange(n_steps_test) * dt_test / T_orb_test

    axes[0].plot(times_test, (r_e - r0_test) / r0_test * 100, lw=0.8, label=f'dt={dt_test}')
    axes[1].plot(times_test, (r_v - r0_test) / r0_test * 100, lw=0.8, label=f'dt={dt_test}')

axes[0].set_title(f'Symplectic Euler at r={r0_test}')
axes[0].set_xlabel('Orbits'); axes[0].set_ylabel('Radius drift (%)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].set_title(f'Velocity Verlet at r={r0_test}')
axes[1].set_xlabel('Orbits'); axes[1].set_ylabel('Radius drift (%)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Orbital period at r={r0_test}: T = {T_orb_test:.3f}")
print(f"Verlet converges much faster: dt=0.001 is already excellent.")
print(f"Euler needs dt < 0.0005 for comparable accuracy.")

## Corrected N-body integrator (velocity Verlet)

Replace symplectic Euler with velocity Verlet for the multi-body case.

In [ ]:
def nbody_simulate_verlet(positions, velocities, masses, force_fn, dt, n_steps,
                          measurement_times=None, labels=None):
    """Velocity Verlet N-body integrator with arbitrary pairwise forces.

    Second-order symplectic: half-kick, drift, recompute forces, half-kick.
    """
    pos = positions.astype(np.float64).copy()
    vel = velocities.astype(np.float64).copy()
    n = len(pos)
    if labels is None:
        labels = np.ones(n)
    time = 0.0

    if measurement_times is None:
        measurement_times = [n_steps * dt]
    measurement_times = sorted(measurement_times)

    rec_pos = [[] for _ in range(n)]
    rec_vel = [[] for _ in range(n)]
    t_idx = 0

    def _record():
        for i in range(n):
            rec_pos[i].append(pos[i].tolist())
            rec_vel[i].append(vel[i].tolist())

    if measurement_times and measurement_times[0] == 0.0:
        _record()
        t_idx = 1

    def _compute_accelerations():
        acc = np.zeros_like(pos)
        for i in range(n):
            for j in range(i + 1, n):
                r_vec = pos[i] - pos[j]
                r_mag = np.linalg.norm(r_vec)
                if r_mag < 1e-12:
                    continue
                f_ij = force_fn(r_vec, r_mag, masses[i], masses[j],
                                labels[i], labels[j])
                acc[i] += f_ij / masses[i]
                acc[j] -= f_ij / masses[j]
        return acc

    acc = _compute_accelerations()

    for _ in range(n_steps):
        if t_idx >= len(measurement_times):
            break

        vel += 0.5 * dt * acc     # half kick
        pos += dt * vel           # full drift
        acc = _compute_accelerations()  # recompute at new positions
        vel += 0.5 * dt * acc     # half kick

        time += dt

        if time >= measurement_times[t_idx] - dt * 0.5:
            _record()
            t_idx += 1

    return {
        'measurement_times': measurement_times[:len(rec_pos[0])],
        'positions': rec_pos,
        'velocities': rec_vel,
    }

## Corrected orbital demo

Sun + planets at various radii, using velocity Verlet with a smaller timestep.
Now inner planets should orbit stably and outer planets (with their very long periods)
should show minimal motion over the simulation time.

To actually see the outer planets "escape," we would need to run for their orbital
timescale ($T \sim 200$ time units), but that's too expensive for a demo.  Instead,
we show that their orbital velocity is tiny compared to inner planets.

In [ ]:
lam = 5.0
M_sun = 50.0
m_planet = 0.01  # test mass to minimize planet-planet perturbations
n_planets = 8
dt_orb = 0.0005  # small dt for accuracy
n_steps_orb = 40000  # T_total = 20

radii = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 13.0, 16.0])
angles = np.linspace(0, 2 * np.pi, n_planets, endpoint=False)

positions_orb = np.zeros((1 + n_planets, 2))
velocities_orb = np.zeros((1 + n_planets, 2))
masses_orb = np.concatenate([[M_sun], np.full(n_planets, m_planet)])

print("Planet setup:")
for idx in range(n_planets):
    r = radii[idx]
    positions_orb[1 + idx] = [r * np.cos(angles[idx]), r * np.sin(angles[idx])]
    F_r = G_scr * M_sun * m_planet / r**2 * np.exp(-r / lam)
    v_circ = np.sqrt(F_r * r / m_planet)
    T_orb = 2 * np.pi * r / v_circ
    velocities_orb[1 + idx] = v_circ * np.array([-np.sin(angles[idx]),
                                                    np.cos(angles[idx])])
    steps_per_orb = T_orb / dt_orb
    n_orbits_sim = n_steps_orb * dt_orb / T_orb
    print(f"  r={r:5.1f}: v_circ={v_circ:.4f}, T={T_orb:8.1f}, "
          f"steps/orbit={steps_per_orb:8.0f}, orbits in sim={n_orbits_sim:.1f}")

def force_fn_orb(r_vec, r_mag, m_i, m_j, label_i, label_j):
    return screened_inv_sq_force(r_vec, r_mag, m_i, m_j, lam=lam)

result_orb = nbody_simulate_verlet(
    positions_orb, velocities_orb, masses_orb, force_fn_orb,
    dt=dt_orb, n_steps=n_steps_orb,
    measurement_times=[0.0] + list(np.arange(dt_orb, n_steps_orb * dt_orb, dt_orb * 40)),
)

traj = np.array(result_orb['positions'])
times_orb = np.array(result_orb['measurement_times'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
colors = plt.cm.viridis(np.linspace(0, 1, n_planets))

ax = axes[0]
ax.plot(0, 0, 'o', ms=12, color='orange', zorder=10, label='Sun')
for idx in range(n_planets):
    t = traj[1 + idx]
    ax.plot(t[:, 0], t[:, 1], '-', lw=0.8, alpha=0.7, color=colors[idx])
    ax.plot(t[0, 0], t[0, 1], 'o', ms=4, color=colors[idx])
theta_circle = np.linspace(0, 2 * np.pi, 100)
ax.plot(lam * np.cos(theta_circle), lam * np.sin(theta_circle),
        'r--', lw=1.5, alpha=0.5, label=f'$\\lambda = {lam}$')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'Orbits (Verlet, dt={dt_orb})')
ax.set_aspect('equal'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
for idx in range(n_planets):
    t = traj[1 + idx]
    r_t = np.linalg.norm(t, axis=1)
    ax.plot(times_orb[:len(r_t)], r_t, '-', lw=1, color=colors[idx],
            label=f'r={radii[idx]:.0f}')
ax.axhline(lam, color='r', ls='--', lw=1.5, alpha=0.5, label=f'$\\lambda$={lam}')
ax.set_xlabel('Time'); ax.set_ylabel('Orbital radius')
ax.set_title('Radius vs time')
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

ax = axes[2]
for idx in range(n_planets):
    t = traj[1 + idx]
    r_t = np.linalg.norm(t, axis=1)
    drift = (r_t - radii[idx]) / radii[idx] * 100
    ax.plot(times_orb[:len(drift)], drift, '-', lw=1, color=colors[idx],
            label=f'r={radii[idx]:.0f}')
ax.axhline(0, color='gray', ls=':', lw=0.5)
ax.set_xlabel('Time'); ax.set_ylabel('Radius drift (%)')
ax.set_title('Relative radius drift')
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## The screening signature: circular velocity vs radius

The key observable for the agent: $v_{\rm circ}(r)$ drops off exponentially
for $r > \lambda$, unlike pure $1/r^2$ gravity where $v_{\rm circ} \propto 1/\!\sqrt{r}$.

In [ ]:
r_plot = np.linspace(0.5, 20.0, 200)
F_screened = G_scr * M_sun * m_planet / r_plot**2 * np.exp(-r_plot / lam)
F_pure = G_scr * M_sun * m_planet / r_plot**2

v_circ_screened = np.sqrt(np.maximum(F_screened * r_plot / m_planet, 0))
v_circ_pure = np.sqrt(np.maximum(F_pure * r_plot / m_planet, 0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(r_plot, v_circ_screened, 'b-', lw=2, label=f'Screened ($\\lambda={lam}$)')
ax.plot(r_plot, v_circ_pure, 'k:', lw=1.5, label='Pure $1/r^2$')
ax.axvline(lam, color='r', ls='--', lw=1.5, alpha=0.5, label=f'$\\lambda={lam}$')
ax.set_xlabel('$r$'); ax.set_ylabel('$v_{\\rm circ}$')
ax.set_title('Circular velocity vs radius')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(r_plot, v_circ_screened / v_circ_pure, 'b-', lw=2)
ax.axhline(1, color='gray', ls=':', lw=0.5)
ax.axvline(lam, color='r', ls='--', lw=1.5, alpha=0.5, label=f'$\\lambda={lam}$')
ax.set_xlabel('$r$')
ax.set_ylabel('$v_{\\rm circ}^{\\rm screened} / v_{\\rm circ}^{\\rm pure}$')
ax.set_title('Ratio: screened / pure')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At r = lambda = {lam}: ratio = {np.exp(-lam/(2*lam)):.3f} = exp(-0.5)")
print(f"At r = 2*lambda = {2*lam}: ratio = {np.exp(-2*lam/(2*lam)):.3f} = exp(-1)")
print(f"At r = 3*lambda = {3*lam}: ratio = {np.exp(-3*lam/(2*lam)):.3f} = exp(-1.5)")

## Recommendation for `stress_tests.ipynb`

1. **Replace the N-body integrator** with velocity Verlet (`nbody_simulate_verlet` above).
   This gives second-order accuracy and symplectic energy conservation.

2. **Use a smaller timestep** ($\Delta t = 0.0005$) for the inner planets.
   With Verlet, this is sufficient even at $r = 1$.

3. **Use test-mass planets** ($m_{\rm planet} = 0.01$) to eliminate planet-planet
   perturbations that complicate the picture.

4. The outer planets ($r \gg \lambda$) don't visibly fly off because their orbital
   period is $\sim 100$x longer than the simulation time.  The screening is best
   visualized through the **$v_{\rm circ}(r)$ curve** rather than trajectory plots.